# 07 — Bayesian Portfolio Optimization

## 1. Title and objective

**Objective.** Compare equal-weight, traditional mean-variance, minimum-volatility, and Bayesian portfolio allocation across the major technology stock universe used in this project: **AAPL, AMZN, FB, GOOG, MSFT, NFLX, and NVDA**.

This notebook is educational. It demonstrates how portfolio allocations change when expected-return and volatility uncertainty is propagated through an optimizer instead of replaced by a single point estimate.


## Educational disclaimer

This notebook is **not investment advice**, a recommendation, or a solicitation to buy or sell any security. The analysis is based on historical data, simplifying assumptions, and research-oriented models. Portfolio weights shown here are for learning and model comparison only; they should not be interpreted as suitable allocations for any individual investor.


## 2. The portfolio optimization problem

Portfolio optimization asks how capital should be allocated across assets given objectives and constraints. In this notebook all optimized portfolios are **long-only** and weights sum to one.

- **Traditional optimization uses point estimates.** Mean-variance optimization usually plugs in one sample mean vector and one sample covariance matrix, then solves for the portfolio with the highest Sharpe-like ratio or the lowest volatility.
- **Point estimates are noisy.** Daily expected returns are especially difficult to estimate because the signal is small relative to daily volatility. A stock can appear attractive because of sampling noise rather than a persistent return advantage.
- **Bayesian optimization propagates uncertainty.** Instead of optimizing once, the Bayesian workflow optimizes once per posterior draw of expected returns and volatilities. This creates a posterior distribution of portfolio weights, showing both a central allocation and uncertainty around that allocation.


## Setup: imports, paths, and reproducibility settings

The path setup below allows the notebook to run from either the repository root or from the `notebooks/` directory. Figures are saved under `reports/figures`.


In [ ]:
from pathlib import Path
import sys

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

NOTEBOOK_PATH = Path.cwd().resolve()
ROOT = NOTEBOOK_PATH if (NOTEBOOK_PATH / "src").exists() else NOTEBOOK_PATH.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.bayesian_models import (
    build_hierarchical_student_t_return_model,
    prepare_returns_for_bayesian_model,
    sample_model,
)
from src.config import (
    DUCKDB_PATH,
    FIGURES_DIR,
    RAW_CSV_PATH,
    STOCK_SYMBOLS,
    TRADING_DAYS_PER_YEAR,
)
from src.model_io import (
    POSTERIOR_SAMPLES_DIR,
    TABLES_DIR,
    ensure_model_dirs,
    load_inference_data,
    save_inference_data,
    save_summary_table,
)
from src.portfolio import (
    bayesian_portfolio_simulation,
    compute_equal_weight_portfolio,
    estimate_mean_covariance,
    historical_var_cvar,
    optimize_max_sharpe_long_only,
    optimize_min_volatility_long_only,
    portfolio_performance,
    summarize_portfolio_weights,
)
from src.sql_utils import initialize_database, query_to_df, run_sql_pipeline

RANDOM_SEED = 42
DRAWS = 1000
TUNE = 1000
CHAINS = 4
TARGET_ACCEPT = 0.90
ALPHA = 0.05
SYMBOLS = list(STOCK_SYMBOLS)

POSTERIOR_DIR = POSTERIOR_SAMPLES_DIR
ensure_model_dirs()
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

RAW_CSV_FOR_NOTEBOOK = RAW_CSV_PATH if RAW_CSV_PATH.exists() else ROOT / "tech_stocks.csv"

sns.set_theme(style="whitegrid", context="notebook")
np.random.seed(RANDOM_SEED)

print(f"Project root:      {ROOT}")
print(f"DuckDB path:       {DUCKDB_PATH}")
print(f"Raw CSV path:      {RAW_CSV_FOR_NOTEBOOK}")
print(f"Posterior dir:     {POSTERIOR_DIR}")
print(f"Figures dir:       {FIGURES_DIR}")
print(f"Tables dir:        {TABLES_DIR}")


## 3. Load `portfolio_returns_wide` from DuckDB

The SQL pipeline creates a restricted, aligned daily log-return panel named `portfolio_returns_wide`. Each row is a date and each selected stock has one return column.


In [ ]:
con = initialize_database(raw_csv_path=RAW_CSV_FOR_NOTEBOOK, db_path=DUCKDB_PATH)
run_sql_pipeline(con)

portfolio_returns_wide = query_to_df(
    con,
    """
    SELECT
        date,
        AAPL,
        AMZN,
        FB,
        GOOG,
        MSFT,
        NFLX,
        NVDA
    FROM portfolio_returns_wide
    ORDER BY date
    """,
)
portfolio_returns_wide["date"] = pd.to_datetime(portfolio_returns_wide["date"])

display(portfolio_returns_wide.head())
print(f"Loaded {len(portfolio_returns_wide):,} aligned daily observations.")


## 4. Prepare the return matrix

The optimization universe is fixed to AAPL, AMZN, FB, GOOG, MSFT, NFLX, and NVDA. The notebook drops incomplete rows so all portfolios are evaluated over the same dates.


## Portfolio finance conventions

Portfolio calculations use aligned daily log-return columns for expected return, covariance, and correlation estimation. For interpretability, portfolio loss metrics are described like simple-return losses when discussing downside thresholds. Annualized return is `mean daily log return × 252`; annualized volatility is `daily volatility × sqrt(252)`.

The reported Sharpe-like metric is `(annualized return - risk_free_rate) / annualized volatility` with `risk_free_rate = 0` for simplicity, so it is a descriptive ranking statistic rather than a formal Sharpe ratio. Historical 5% VaR is the 5th percentile of portfolio daily returns and is interpreted as a lower-tail loss threshold. Historical 5% CVaR is the average portfolio return on days at or below that VaR threshold. Correlation is central: when tech stocks are highly correlated, adding more tickers may not add much diversification because shared market, growth, and sentiment exposures can move the portfolio together.



In [ ]:
return_matrix = (
    portfolio_returns_wide
    .set_index("date")
    .loc[:, SYMBOLS]
    .apply(pd.to_numeric, errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
    .dropna(how="any")
)

mean_daily_returns, covariance_daily = estimate_mean_covariance(return_matrix)
historical_corr = return_matrix.corr().loc[SYMBOLS, SYMBOLS]

print(f"Return matrix shape: {return_matrix.shape[0]:,} days x {return_matrix.shape[1]} stocks")
display(return_matrix.describe().T)
display(historical_corr.style.format("{:.2f}"))


## Portfolio metric helpers

All methods are evaluated with the same historical daily return matrix. The Sharpe-like ratio uses a zero risk-free rate for comparability with the project utility functions.


In [ ]:
def max_drawdown(portfolio_returns: pd.Series) -> float:
    """Compute max drawdown from daily log returns."""
    clean_returns = pd.to_numeric(portfolio_returns, errors="coerce").dropna()
    if clean_returns.empty:
        return np.nan
    wealth_index = np.exp(clean_returns.cumsum())
    drawdown = wealth_index / wealth_index.cummax() - 1.0
    return float(drawdown.min())


def portfolio_returns_from_weights(weights, returns_df=return_matrix) -> pd.Series:
    """Compute daily log portfolio returns for a weight vector."""
    weights_array = np.asarray(weights, dtype="float64")
    return pd.Series(
        returns_df.to_numpy(dtype="float64") @ weights_array,
        index=returns_df.index,
        name="portfolio_return",
    )


def evaluate_weighted_portfolio(name: str, weights, returns_df=return_matrix) -> dict[str, float | str]:
    """Evaluate one static-weight portfolio on historical returns."""
    weights_array = np.asarray(weights, dtype="float64")
    daily_returns = portfolio_returns_from_weights(weights_array, returns_df)
    performance = portfolio_performance(
        weights_array,
        mean_daily_returns.loc[SYMBOLS],
        covariance_daily.loc[SYMBOLS, SYMBOLS],
        trading_days=TRADING_DAYS_PER_YEAR,
    )
    var, cvar = historical_var_cvar(daily_returns, alpha=ALPHA)
    return {
        "method": name,
        **performance,
        "max_drawdown": max_drawdown(daily_returns),
        f"var_{int(ALPHA * 100)}pct": var,
        f"cvar_{int(ALPHA * 100)}pct": cvar,
    }


def allocation_table(weights, method: str) -> pd.DataFrame:
    """Return a tidy allocation table for display."""
    return pd.DataFrame(
        {
            "method": method,
            "symbol": SYMBOLS,
            "weight": np.asarray(weights, dtype="float64"),
        }
    ).sort_values("weight", ascending=False, ignore_index=True)


def format_metric_table(df: pd.DataFrame):
    """Format a portfolio-comparison table as percentages where appropriate."""
    formatters = {
        "annualized_return": "{:.2%}",
        "annualized_volatility": "{:.2%}",
        "sharpe_like_ratio": "{:.2f}",
        "max_drawdown": "{:.2%}",
        f"var_{int(ALPHA * 100)}pct": "{:.2%}",
        f"cvar_{int(ALPHA * 100)}pct": "{:.2%}",
    }
    return df.style.format({k: v for k, v in formatters.items() if k in df.columns})


## 5. Benchmark 1: Equal-weight portfolio

The equal-weight portfolio assigns the same weight to each stock. It is a simple benchmark because it uses no expected-return or covariance estimates to set allocations.


In [ ]:
equal_weights = np.repeat(1 / len(SYMBOLS), len(SYMBOLS))
equal_weight_returns = compute_equal_weight_portfolio(return_matrix)
equal_weight_metrics = evaluate_weighted_portfolio("Equal weight", equal_weights)

display(allocation_table(equal_weights, "Equal weight").style.format({"weight": "{:.2%}"}))
display(format_metric_table(pd.DataFrame([equal_weight_metrics])))


## 6. Benchmark 2: Traditional mean-variance max-Sharpe portfolio

This optimizer uses the sample daily mean vector and sample daily covariance matrix as if they were known. The portfolio is long-only and maximizes a Sharpe-like ratio with a zero risk-free rate.


In [ ]:
traditional_max_sharpe_weights, traditional_max_sharpe_performance = optimize_max_sharpe_long_only(
    mean_daily_returns.loc[SYMBOLS],
    covariance_daily.loc[SYMBOLS, SYMBOLS],
)
traditional_max_sharpe_weights = traditional_max_sharpe_weights.loc[SYMBOLS]
traditional_max_sharpe_metrics = evaluate_weighted_portfolio(
    "Traditional max Sharpe",
    traditional_max_sharpe_weights,
)

traditional_max_sharpe_allocation = allocation_table(
    traditional_max_sharpe_weights,
    "Traditional max Sharpe",
)
display(traditional_max_sharpe_allocation.style.format({"weight": "{:.2%}"}))
display(format_metric_table(pd.DataFrame([traditional_max_sharpe_metrics])))


## 7. Benchmark 3: Minimum-volatility portfolio

The minimum-volatility optimizer uses the sample covariance matrix but does not use expected returns in the objective. This often creates a more diversified low-risk benchmark than a return-seeking max-Sharpe optimizer.


In [ ]:
minimum_volatility_weights, _ = optimize_min_volatility_long_only(
    covariance_daily.loc[SYMBOLS, SYMBOLS]
)
minimum_volatility_weights = minimum_volatility_weights.loc[SYMBOLS]
minimum_volatility_metrics = evaluate_weighted_portfolio(
    "Minimum volatility",
    minimum_volatility_weights,
)

minimum_volatility_allocation = allocation_table(
    minimum_volatility_weights,
    "Minimum volatility",
)
display(minimum_volatility_allocation.style.format({"weight": "{:.2%}"}))
display(format_metric_table(pd.DataFrame([minimum_volatility_metrics])))


## 8. Bayesian portfolio optimization

The Bayesian workflow uses posterior draws of stock-level expected daily return (`stock_mu`) and daily volatility (`stock_sigma`) from the hierarchical Student-t return model. For each posterior draw, the notebook combines those volatility draws with the historical correlation matrix to build a covariance matrix, then solves a long-only max-Sharpe optimization.

This produces a posterior distribution over portfolio weights instead of a single optimizer output.


In [ ]:
def load_or_fit_return_model() -> az.InferenceData:
    """Load a cached Bayesian return model or fit one for the portfolio universe."""
    preferred_cache_paths = [
        POSTERIOR_DIR / "return_model.nc",
    ]
    for cache_path in preferred_cache_paths:
        if cache_path.exists():
            candidate = load_inference_data(cache_path)
            if {"stock_mu", "stock_sigma"}.issubset(set(candidate.posterior.data_vars)):
                print(f"Loaded cached Bayesian return model: {cache_path}")
                return candidate

    print("No compatible cached Bayesian return model found; fitting a new model for the portfolio universe.")
    long_returns = return_matrix.reset_index().melt(
        id_vars="date",
        value_vars=SYMBOLS,
        var_name="symbol",
        value_name="log_return",
    )
    returns, stock_idx, symbol_lookup, _ = prepare_returns_for_bayesian_model(
        long_returns,
        symbols=SYMBOLS,
    )
    ordered_symbols = [symbol_lookup[idx] for idx in sorted(symbol_lookup)]
    if ordered_symbols != SYMBOLS:
        raise ValueError(f"Unexpected symbol order: {ordered_symbols}")

    model = build_hierarchical_student_t_return_model(
        returns=returns,
        stock_idx=stock_idx,
        n_stocks=len(SYMBOLS),
    )
    idata = sample_model(
        model,
        draws=DRAWS,
        tune=TUNE,
        chains=CHAINS,
        target_accept=TARGET_ACCEPT,
    )
    cache_path = POSTERIOR_DIR / "return_model.nc"
    save_inference_data(idata, cache_path)
    print(f"Saved Bayesian return model cache: {cache_path}")
    return idata


def posterior_draws_to_frame(idata: az.InferenceData, variable: str, symbols: list[str]) -> pd.DataFrame:
    """Extract stock-level posterior draws into a draw-by-symbol DataFrame."""
    draws = idata.posterior[variable]
    stacked = draws.stack(sample=("chain", "draw"))
    stock_dims = [dim for dim in stacked.dims if dim != "sample"]
    if len(stock_dims) != 1:
        raise ValueError(f"Expected one stock dimension for {variable}, found {draws.dims}")
    values = stacked.transpose("sample", stock_dims[0]).values
    if values.shape[1] < len(symbols):
        raise ValueError(f"{variable} has {values.shape[1]} stocks, expected at least {len(symbols)}")
    return pd.DataFrame(values[:, : len(symbols)], columns=symbols)


return_model_idata = load_or_fit_return_model()
posterior_return_draws = posterior_draws_to_frame(return_model_idata, "stock_mu", SYMBOLS)
posterior_vol_draws = posterior_draws_to_frame(return_model_idata, "stock_sigma", SYMBOLS)

print(f"Posterior return draw matrix: {posterior_return_draws.shape}")
print(f"Posterior volatility draw matrix: {posterior_vol_draws.shape}")


In [ ]:
bayesian_optimization_draws = bayesian_portfolio_simulation(
    posterior_return_draws=posterior_return_draws,
    posterior_vol_draws=posterior_vol_draws,
    corr_matrix=historical_corr,
    symbol_lookup=SYMBOLS,
)

weight_columns = [f"weight_{symbol}" for symbol in SYMBOLS]
bayesian_weight_draws = bayesian_optimization_draws.loc[:, weight_columns]
bayesian_weight_draws.columns = SYMBOLS

display(bayesian_optimization_draws.head())
print(f"Optimized {len(bayesian_optimization_draws):,} Bayesian posterior portfolios.")


## 9. Bayesian allocation summary

The table below reports the mean, median, and 90% credible interval for each stock's Bayesian optimized weight. It also reports the posterior probability that each stock receives the largest allocation in a draw.


In [ ]:
bayesian_weight_summary = summarize_portfolio_weights(bayesian_optimization_draws)
largest_allocation_symbol = bayesian_weight_draws.idxmax(axis=1)
probability_largest_allocation = (
    largest_allocation_symbol
    .value_counts(normalize=True)
    .reindex(SYMBOLS, fill_value=0.0)
    .rename("probability_largest_allocation")
)

bayesian_allocation_summary = (
    bayesian_weight_summary
    .rename(columns={"weight_5pct": "credible_interval_5pct", "weight_95pct": "credible_interval_95pct"})
    .join(probability_largest_allocation)
    .reset_index()
)

mean_bayesian_weights = bayesian_weight_summary.loc[SYMBOLS, "mean"]
median_bayesian_weights = bayesian_weight_summary.loc[SYMBOLS, "median"]

format_columns = {
    "mean": "{:.2%}",
    "median": "{:.2%}",
    "credible_interval_5pct": "{:.2%}",
    "credible_interval_95pct": "{:.2%}",
    "probability_largest_allocation": "{:.1%}",
}
bayesian_allocation_summary_path = save_summary_table(
    bayesian_allocation_summary,
    TABLES_DIR / "07_bayesian_allocation_summary.csv",
)
print(f"Saved Bayesian allocation summary to {bayesian_allocation_summary_path}")
display(bayesian_allocation_summary.style.format(format_columns))


## 10. Compare all portfolio methods in one table

The Bayesian max-Sharpe row uses the posterior **mean** Bayesian weight vector as a representative uncertainty-aware allocation summary. The posterior distribution itself is shown in the next section.


In [ ]:
bayesian_mean_metrics = evaluate_weighted_portfolio(
    "Bayesian max Sharpe",
    mean_bayesian_weights.loc[SYMBOLS],
)

comparison_table = pd.DataFrame(
    [
        equal_weight_metrics,
        traditional_max_sharpe_metrics,
        minimum_volatility_metrics,
        bayesian_mean_metrics,
    ]
)

method_weights = pd.DataFrame(
    {
        "Equal weight": equal_weights,
        "Traditional max Sharpe": traditional_max_sharpe_weights.loc[SYMBOLS].to_numpy(dtype="float64"),
        "Minimum volatility": minimum_volatility_weights.loc[SYMBOLS].to_numpy(dtype="float64"),
        "Bayesian max Sharpe": mean_bayesian_weights.loc[SYMBOLS].to_numpy(dtype="float64"),
    },
    index=SYMBOLS,
)

comparison_table_path = save_summary_table(
    comparison_table,
    TABLES_DIR / "07_portfolio_method_comparison.csv",
)
method_weights_path = save_summary_table(
    method_weights.reset_index(names="symbol"),
    TABLES_DIR / "07_portfolio_method_weights.csv",
)
print(f"Saved portfolio method comparison to {comparison_table_path}")
print(f"Saved portfolio method weights to {method_weights_path}")

comparison_display = comparison_table.copy()
display(format_metric_table(comparison_display))
display(method_weights.style.format("{:.2%}"))


## 11. Plot portfolio weights by method

The figure compares static weights from equal-weight, traditional max-Sharpe, minimum-volatility, and Bayesian mean max-Sharpe summaries.


In [ ]:
weights_plot_df = (
    method_weights
    .reset_index(names="symbol")
    .melt(id_vars="symbol", var_name="method", value_name="weight")
)

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(
    data=weights_plot_df,
    x="symbol",
    y="weight",
    hue="method",
    ax=ax,
)
ax.set_title("Portfolio weights by allocation method")
ax.set_xlabel("Symbol")
ax.set_ylabel("Portfolio weight")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
ax.legend(title="Method", loc="upper left", bbox_to_anchor=(1.02, 1.0))
fig.tight_layout()
weights_by_method_path = FIGURES_DIR / "07_portfolio_weights_by_method.png"
fig.savefig(weights_by_method_path, dpi=300, bbox_inches="tight")
weights_by_method_path


## 12. Plot posterior Bayesian allocation uncertainty

The Bayesian optimizer returns a distribution of weights. The plot below shows the posterior median, 90% credible interval, and mean weight for each stock.


In [ ]:
uncertainty_plot_df = bayesian_allocation_summary.sort_values("mean", ascending=True)

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.errorbar(
    uncertainty_plot_df["median"],
    uncertainty_plot_df["symbol"],
    xerr=[
        uncertainty_plot_df["median"] - uncertainty_plot_df["credible_interval_5pct"],
        uncertainty_plot_df["credible_interval_95pct"] - uncertainty_plot_df["median"],
    ],
    fmt="o",
    color="#4C78A8",
    ecolor="#4C78A8",
    elinewidth=2,
    capsize=4,
    label="Median with 90% credible interval",
)
ax.scatter(
    uncertainty_plot_df["mean"],
    uncertainty_plot_df["symbol"],
    color="#F58518",
    marker="D",
    s=55,
    label="Mean",
)
ax.set_xlim(0, max(1.0, uncertainty_plot_df["credible_interval_95pct"].max() * 1.05))
ax.set_title("Posterior uncertainty in Bayesian portfolio weights")
ax.set_xlabel("Portfolio weight")
ax.set_ylabel("Symbol")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
ax.legend(loc="lower right")
fig.tight_layout()
bayesian_uncertainty_path = FIGURES_DIR / "07_bayesian_allocation_uncertainty.png"
fig.savefig(bayesian_uncertainty_path, dpi=300, bbox_inches="tight")
bayesian_uncertainty_path


## 13. Interpretation

- The Bayesian portfolio is typically **less overconfident** than a single traditional max-Sharpe solution because it exposes how weights vary across plausible return and volatility parameter draws.
- The allocations should be interpreted as **uncertainty-aware historical estimates**, not as predictions guaranteed to hold in the future.
- Concentrated weights may indicate noisy expected-return estimates or model sensitivity, not guaranteed future outperformance.
- A wide credible interval means the optimizer frequently changes its preferred allocation when parameter uncertainty is propagated through the portfolio problem.
- These results are for education and model comparison only, and they are **not investment advice**.


## 14. Saved figures

The notebook saves these figures to `reports/figures`:

- `07_portfolio_weights_by_method.png`
- `07_bayesian_allocation_uncertainty.png`


In [ ]:
display(Markdown("### Figure outputs"))
for figure_path in [weights_by_method_path, bayesian_uncertainty_path]:
    print(figure_path)

display(Markdown("### Table outputs"))
for table_path in [
    bayesian_allocation_summary_path,
    comparison_table_path,
    method_weights_path,
]:
    print(table_path)
